In [2]:
import tensorflow as tf
import numpy as np

2025-07-02 14:07:25.481720: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-02 14:07:25.499563: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-02 14:07:25.610780: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-02 14:07:25.687311: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751465245.769201   66306 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751465245.78

In [3]:
from migration.datasets import create_AIS_dataset
inputs, targets, _, _, _, lengths, mean =  create_AIS_dataset('../../data/ct_2017010203_10_20/ct_2017010203_10_20_train.pkl', 
                   '../../data/ct_2017010203_10_20/mean.pkl',
                   32,
                   99999, # not used lol
                   300,
                   300, 
                   30,
                   72, 
                   shuffle=False,
                   repeat=False)

Instructions for updating:
Use output_signature instead
Instructions for updating:
tf.py_func is deprecated in TF V2. Instead, there are two
    options available in V2.
    - tf.py_function takes a python function which manipulates tf eager
    tensors instead of numpy arrays. It's easy to convert a tf eager tensor to
    an ndarray (just call tensor.numpy()) but having access to eager tensors
    means `tf.py_function`s can use accelerators such as GPUs as well as
    being differentiable using a gradient tape.
    - tf.numpy_function maintains the semantics of the deprecated tf.py_func
    (it is not differentiable, and manipulates numpy arrays). It drops the
    stateful argument making all functions stateful.
    


2025-07-02 14:07:40.515428: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
2025-07-02 14:07:40.880449: E tensorflow/core/util/util.cc:131] oneDNN supports DT_BOOL only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


In [4]:
seq_lengths = lengths
batch_size = tf.shape(input=seq_lengths)[0]
max_seq_len = tf.reduce_max(input_tensor=seq_lengths)

# shape (t, B) of 1 and 0
seq_mask = tf.transpose(
        a=tf.sequence_mask(seq_lengths, maxlen=max_seq_len, dtype=tf.float32),
        perm=[1, 0])

In [5]:
latent_size = 64
_DEFAULT_INITIALIZERS = {"w": tf.keras.initializers.VarianceScaling(scale=1.0, mode="fan_avg", distribution="uniform",seed=111),
                         "b": tf.zeros_initializer()}

In [21]:
rnn_cell = tf.keras.layers.LSTMCell(latent_size, kernel_initializer=_DEFAULT_INITIALIZERS['w'])

# has projection option but is not used
# activation is tahn
# 
# tf.compat.v1.nn.rnn_cell.LSTMCell(latent_size, initializer=_DEFAULT_INITIALIZERS["w"])

In [23]:
rnn_cell.get_initial_state(32)

[<tf.Tensor: shape=(32, 64), dtype=float32, numpy=
 array([[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)>,
 <tf.Tensor: shape=(32, 64), dtype=float32, numpy=
 array([[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)>]

In [26]:
# The zero state of the model: LSTM state, output of the latent_feat_extractor perceptron
rnn_state = rnn_cell.get_initial_state(32)
prev_latent_encoded = tf.zeros((32, 64), dtype=tf.float32)


inputs_encoded = tf.random.uniform((32,64), seed=1)
targets_encoded = tf.random.uniform((32,64), seed=1)
rnn_inputs = tf.concat([inputs_encoded, prev_latent_encoded], axis=1)

In [27]:
rnn_inputs

<tf.Tensor: shape=(32, 128), dtype=float32, numpy=
array([[0.3037281 , 0.6981312 , 0.43454254, ..., 0.        , 0.        ,
        0.        ],
       [0.11873066, 0.22871375, 0.10807776, ..., 0.        , 0.        ,
        0.        ],
       [0.43794978, 0.03318262, 0.24484766, ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.5276431 , 0.6229882 , 0.0233959 , ..., 0.        , 0.        ,
        0.        ],
       [0.9583149 , 0.45939898, 0.6769912 , ..., 0.        , 0.        ,
        0.        ],
       [0.69264436, 0.5421194 , 0.9338697 , ..., 0.        , 0.        ,
        0.        ]], dtype=float32)>

In [28]:
rnn_state

[<tf.Tensor: shape=(32, 64), dtype=float32, numpy=
 array([[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)>,
 <tf.Tensor: shape=(32, 64), dtype=float32, numpy=
 array([[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)>]

In [29]:
rnn_out, new_rnn_state = rnn_cell(rnn_inputs, rnn_state)

In [30]:
rnn_out

<tf.Tensor: shape=(32, 64), dtype=float32, numpy=
array([[ 0.0416259 , -0.06208875,  0.0123661 , ...,  0.03817587,
         0.05555509,  0.06718442],
       [ 0.01499553, -0.06219564, -0.03819848, ..., -0.00502684,
        -0.07485715, -0.01959298],
       [-0.01609482, -0.1123019 , -0.05041137, ...,  0.04715925,
         0.0431211 ,  0.00823885],
       ...,
       [ 0.03087194, -0.00573284, -0.06188539, ..., -0.00258076,
        -0.03528319,  0.00457118],
       [ 0.06819934, -0.09858456,  0.00076405, ...,  0.05478743,
        -0.06185243,  0.01414165],
       [ 0.09930357, -0.07063153, -0.00291835, ...,  0.00102115,
        -0.0546855 ,  0.05277814]], dtype=float32)>

In [31]:
rnn_state

[<tf.Tensor: shape=(32, 64), dtype=float32, numpy=
 array([[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)>,
 <tf.Tensor: shape=(32, 64), dtype=float32, numpy=
 array([[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)>]